In [ ]:
from glob import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, precision_score, recall_score, accuracy_score
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier
from sklearn.svm import LinearSVR
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.multiclass import OneVsRestClassifier
from sklearn.datasets import fetch_openml
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import ConfusionMatrixDisplay



fashion_path_train = glob("../Fashion_MNIST/fashion-mnist_train.csv")
fashion_path_test = glob("../Fashion_MNIST/fashion-mnist_test.csv")
# fish_path
fashion_data_train_ = pd.read_csv(fashion_path_train[0])
fashion_data_test_ = pd.read_csv(fashion_path_test[0])

# fashion_data_train_.shape
# fashion_data_test_.shape

# fashion_data_train_.isnull().sum().sum()
# fashion_data_train_.isna().sum().sum()
# fashion_data_test_.isnull().sum().sum()
# fashion_data_test_.isna().sum().sum()


### Correlation entre données

In [ ]:
correl = fashion_data_train_.corr()

correl_target = abs(correl["label"])

best_feats = correl_target[correl_target > 0.5]

for i in range(51):
    print("Higher than",round(0.5 + i*0.01, 2), "correlation: # of Pixels:",len(correl_target[correl_target > (0.5 + i*0.01)])-1)
    if len(correl_target[correl_target > (0.5 + i*0.01)]) == 1:
        break



### Function to rename label

In [ ]:
def lab_to_name(label):
    labeled = label.copy()
    mapping = {0 :'T-Shirt/Top',
    1 :'Trouser',
    2 :'Women-Shirt',
    3 :'Dress',
    4 :'Pull-Over',
    5 :'Sandal',
    6 :'Men-Shirt',
    7 :'Sneakers',
    8 :'Hand Bag',
    9 :'Boot'}
    labeled = label.map(mapping)
    return labeled



### Train & test

In [ ]:
from sklearn.preprocessing import StandardScaler

X_tra__ = fashion_data_train_.drop(['label'], axis = 1)
X_tes__ = fashion_data_test_.drop(['label'], axis = 1)
y_train = fashion_data_train_["label"]
y_test = fashion_data_test_["label"]

scaler = StandardScaler()
X_train = scaler.fit_transform(X_tra__)
X_test = scaler.transform(X_tes__)

In [ ]:
fashion_data_train_.label.unique()

### Vérification

In [ ]:
fig, axes = plt.subplots(5, 5, figsize=(20,20))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_train.iloc[i].to_numpy().reshape(28,28), cmap='gray')
    ax.set_title(f"Label -- {y_train.iloc[i]} : {lab_to_name(y_train)[i]}")
    ax.axis('off')
plt.show()


### Bagging Classifier

In [49]:
KNN = KNeighborsClassifier(n_neighbors=3)
DTC = DecisionTreeClassifier(max_depth=5, criterion='gini',random_state=42)
OVR_KN = OneVsRestClassifier(KNeighborsClassifier(n_neighbors=3))
OVR_DT = OneVsRestClassifier(DecisionTreeClassifier(max_depth=5, criterion='gini',random_state=42))

base_m = [KNN, DTC, OVR_KN]

for m in base_m:
    print(f"Methods : {m}")
    model_bagging = BaggingClassifier(estimator = m, n_estimators=5, random_state=42)
    model_bagging.fit(X_train, y_train)
    y_pred_bagging = model_bagging.predict(X_test)
    print(f"Accuracy : {accuracy_score(y_test, y_pred_bagging)}")
    print(classification_report(y_test, y_pred_bagging))
    confusion_m = confusion_matrix(y_test, y_pred_bagging, labels=model_bagging.classes_)
    disp = ConfusionMatrixDisplay(confusion_matrix=confusion_m, display_labels=model_bagging.classes_)
    disp.plot(cmap=plt.cm.Reds)
    plt.title('Confusion Matrix')
    plt.xlabel('Predicted label')
    plt.ylabel('True label')
    plt.show()






Methods : KNeighborsClassifier(n_neighbors=3)


KeyboardInterrupt: 

### Find Best Parameters

In [ ]:
train_res = {}
test_res = {}

model_test_DTC = DecisionTreeClassifier()

param_grid = {'max_depth' : np.arange(1,7)}
model_DTC_CV = GridSearchCV(model_test_DTC, param_grid, cv = 5)
model_DTC_CV.fit(X_train, y_train)
print(model_DTC_CV.best_params_)
print(model_DTC_CV.best_score_)

In [ ]:
train_res = {}
test_res = {}

model_test_KNN = KNeighborsClassifier()

param_grid = {'n_neighbors' : np.arange(1,10)}
model_KNN_CV = GridSearchCV(model_test_KNN, param_grid, cv = 5)
model_KNN_CV.fit(X_train, y_train)
print(model_KNN_CV.best_params_)
print(model_KNN_CV.best_score_)

In [50]:
KNN = KNeighborsClassifier(n_neighbors=4)
DTC = DecisionTreeClassifier(max_depth=6, criterion='gini',random_state=42)
OVR_KN = OneVsRestClassifier(KNeighborsClassifier(n_neighbors=3))
OVR_DT = OneVsRestClassifier(DecisionTreeClassifier(max_depth=5, criterion='gini',random_state=42))

base_m_new = [KNN, DTC, OVR_KN]

train_res = {}
test_res = {}

estimators = [2, 4, 5, 8]

for estim in estimators:
    for m in base_m:
        print(f"Methods : {m}")
        model_bagging = BaggingClassifier(estimator = m, n_estimators=estim, random_state=42)
        model_bagging.fit(X_train, y_train)
        y_pred_bagging = model_bagging.predict(X_test)
        train_res[estim] = model_bagging.score(X_train, y_train)
        train_res[f" Accuracy {estim}"] = model_bagging.score(X_train, y_train)
        test_res[f" Accuracy {estim}"] = model_bagging.score(X_test, y_test)

# train_res_df = pd.DataFrame(train_res)
# test_res_df = pd.DataFrame(test_res)
#
# train_res_df = train_res_df

Methods : KNeighborsClassifier(n_neighbors=3)
Methods : DecisionTreeClassifier(max_depth=5, random_state=42)
Methods : OneVsRestClassifier(estimator=KNeighborsClassifier(n_neighbors=3))
Methods : KNeighborsClassifier(n_neighbors=3)
Methods : DecisionTreeClassifier(max_depth=5, random_state=42)
Methods : OneVsRestClassifier(estimator=KNeighborsClassifier(n_neighbors=3))
Methods : KNeighborsClassifier(n_neighbors=3)
Methods : DecisionTreeClassifier(max_depth=5, random_state=42)
Methods : OneVsRestClassifier(estimator=KNeighborsClassifier(n_neighbors=3))


KeyboardInterrupt: 

In [52]:
# train_res_df = pd.DataFrame.from_dict(train_res)
# test_res_df = pd.DataFrame.from_dict(test_res)
#
# train_res_df.sort_values(by='', ascending=False)
train_res

{2: 0.9152166666666667,
 ' Accuracy 2': 0.9152166666666667,
 4: 0.9218166666666666,
 ' Accuracy 4': 0.9218166666666666,
 5: 0.7262666666666666,
 ' Accuracy 5': 0.7262666666666666}